### 1. Basic Tasks

In [0]:
-- 1. Create a catalog cyntexa_dev and a schema sales within it.
create catalog if not exists cyntexa_dev;
create schema if not exists cyntexa_dev.sales

In [0]:
-- 2. Create a managed table sales.orders_raw with at least 5 columns and insert 10 sample rows.
create table if not exists cyntexa_dev.sales.orders_raw(order_id int, customer_id string, product string, amount double, status string)

In [0]:
insert into cyntexa_dev.sales.orders_raw values
    (1, "C1001", "Laptop", 55000.00, "Delivered"),
    (2, "C1002", "Mouse", 1200.00, "Pending"),
    (3, "C1003", "Keyboard", 2500.00, "Shipped"),
    (4, "C1004", "Monitor", 30000.00,"Delivered"),
    (5, "C1005", "Headphones", 3500.00, "Cancelled"),
    (6, "C1001", "USB Cable", 900.00, "Delivered"),
    (7, "C1006", "Laptop Bag", 1800.00, "Pending"),
    (8, "C1007", "Webcam", 5000.00, "Shipped"),
    (9, "C1008", "Printer", 12000.00, "Delivered"),
    (10, "C1009", "SSD", 7000.00, "Pending")

In [0]:
-- 3. Create a view sales.orders_view that selects only completed orders.
create view orders_view as 
select * from cyntexa_dev.sales.orders_raw
where status = "Delivered"

In [0]:
-- 4. (Data Analyst) Explore samples.bakehouse or samples.tpch and run 3 exploratory SELECT queries.
select * from samples.bakehouse.sales_transactions limit 5 

In [0]:
-- Count of transactions by each payment method
select count(transactionID), paymentMethod 
from samples.bakehouse.sales_transactions
group by paymentMethod 
order by count(transactionID)

In [0]:
-- Sales by each franchise
select franchiseID, sum(totalPrice) 
from samples.bakehouse.sales_transactions
group by franchiseID
order by sum(totalPrice) desc

In [0]:
-- total order value per customer
select customerID, sum(totalPrice)
 from samples.bakehouse.sales_transactions
 group by customerID
 order by sum(totalPrice) desc

### 2. Intermediate Tasks

In [0]:
-- Write a SQL UDF that masks the last 4 digits of a customer_id or email column, and apply it in a SELECT against orders_raw.
create function cyntexa_dev.sales.datamask(x string)
returns string
return concat(left(x,1), '****')

In [0]:
select order_id, cyntexa_dev.sales.datamask(customer_id) as customer_id, product, amount, status 
from cyntexa_dev.sales.orders_raw

In [0]:
-- 7. (Data Analyst) Build a second view joining orders_view with a customers table/view and calculate total spend per customer.
create view total_customer_spend as
select customer_id, sum(amount) as total_spent
from cyntexa_dev.sales.orders_raw
group by customer_id 

In [0]:
select * from total_customer_spend

### 3. Advanced Tasks

In [0]:
-- 8. Design a full three-level namespace plan for Cyntexa (catalogs for dev/staging/prod, schemas per business domain) and justify the structure in a short writeup.
create catalog if not exists cyntexa_dev;
create schema if not exists cyntexa_dev.sales;
create schema if not exists cyntexa_dev.products;
create schema if not exists cyntexa_dev.finance;

create catalog if not exists cyntexa_uat;
create schema if not exists cyntexa_uat.sales;
create schema if not exists cyntexa_uat.products;
create schema if not exists cyntexa_uat.finance;

create catalog if not exists cyntexa_prod;
create schema if not exists cyntexa_prod.sales;
create schema if not exists cyntexa_prod.products;
create schema if not exists cyntexa_prod.finance;

The namespace use 3 catalogs(dev, uat and prod) each representing seperate environment. Within each catalof sechemas are organised according to the business domains (sales, product, and finance).

This sturcture provides clear seperation between environments ad business domains. It also makes it easier to manage permisions, as users can be given access to specific catalogs or schema based on their responsibilities.

The 3 level namespace follows this structure :

Catalog.Schema.table

This provide better organization, security, governance and environment isolation for a platform 

9. 
Data Masking Strategy for Cyntexa

An organization should apply data masking to sensitive columns so that users can work with the data they need without exposing confidential information unnecessarily.

For example, an email address such as `john@example.com` could be displayed as `j***@example.com` to users who do not require the full value.

2. Role-Based Access

Different user roles should have different levels of access:

* Data Engineers: Unmasked data where required for data ingestion, cleaning, and transformation.
* Data Analysts: Masked sensitive information. Analysts should generally only see the data required for reporting and analysis.
* Business Users: Highly restricted access with masked sensitive columns.
* Data Governance/Admin Team: Unmasked access when required for governance, auditing, or approved operational purposes.

The principle of least privilege should be followed, meaning users should only receive access to sensitive information when it is necessary for their job.

3. Enforcing Masking with Unity Catalog

Unity Catalog can be used to control access to catalogs, schemas, tables, and sensitive columns through permissions and governance policies. Sensitive columns can be protected using **column-level access controls and dynamic views/row filters or column masks**, depending on the Databricks implementation.

For example, an analyst could have:

SELECT → Allowed

customer_email → Masked

customer_phone → Masked

while an authorized data engineer could receive access to the original values.

This ensures that access to sensitive information is controlled centrally rather than relying on individual users or applications to hide the data.

### Conclusion

The strategy follows the **least-privilege principle**: sensitive data is masked by default and unmasked access is provided only to authorized roles that have a legitimate business requirement. Unity Catalog provides centralized governance and permission management to enforce these controls consistently.


In [0]:
-- 10. (Data Analyst) Using samples.tpch, write a query with at least one CTE and one window function to produce a 'top 5 customers by revenue per region' report.
with customer_total as(
    select c.c_custkey, r.r_regionkey, sum(o.o_totalprice) as total_revenue
    from samples.tpch.customer c  
    join samples.tpch.orders o
    on c.c_custkey = o.o_custkey
    join samples.tpch.region r
    on c.c_nationkey = r.r_regionkey
    group by c.c_custkey, r.r_regionkey
),
customer_rank as (
select c_custkey, r_regionkey, total_revenue,
rank() over (partition by r_regionkey order by total_revenue desc) as rank
from customer_total
)

select * from customer_rank where rank <= 5